# TCM Shape Debug

Run `TemporalContextBlock` with synthetic inputs and print tensor shapes.

This notebook targets `ldm/modules/temporal_modules/tcm.py` directly. The module already prints intermediate tensor shapes during `forward`, so the main thing here is setting up a clean import and providing sample inputs.

In [1]:
print('Hello World')

Hello World


In [2]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "ldm").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))
print(f"repo_root={repo_root}")
print("Open this notebook from the repo root or from scripts/.")

repo_root=/data_ssd/caioseda/projetos/Stable-Diffusion-Seg
Open this notebook from the repo root or from scripts/.


In [3]:
import importlib.util
import torch
import torch.nn.init as init

# tcm.py expects mmcv-style helpers to exist under torch.nn.init.
if not hasattr(init, "constant_init"):
    def constant_init(module, val=0, bias=0):
        init.constant_(module.weight, val)
        if getattr(module, "bias", None) is not None:
            init.constant_(module.bias, bias)
    init.constant_init = constant_init

if not hasattr(init, "kaiming_init"):
    def kaiming_init(module, mode="fan_in", nonlinearity="relu", bias=0, distribution="normal"):
        if distribution == "uniform":
            init.kaiming_uniform_(module.weight, mode=mode, nonlinearity=nonlinearity)
        else:
            init.kaiming_normal_(module.weight, mode=mode, nonlinearity=nonlinearity)
        if getattr(module, "bias", None) is not None:
            init.constant_(module.bias, bias)
    init.kaiming_init = kaiming_init

tcm_path = repo_root / "ldm" / "modules" / "temporal_modules" / "tcm.py"
spec = importlib.util.spec_from_file_location("tcm_debug_module", tcm_path)
tcm_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tcm_module)

TemporalContextBlock = tcm_module.TemporalContextBlock
print(f"Loaded TemporalContextBlock from {tcm_path}")
print(f"torch={torch.__version__}, cuda_available={torch.cuda.is_available()}")

/home/caioseda/miniconda3/envs/sdseg-cpython/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded TemporalContextBlock from /data_ssd/caioseda/projetos/Stable-Diffusion-Seg/ldm/modules/temporal_modules/tcm.py
torch=1.11.0, cuda_available=True


In [5]:
def run_tcm_case(batch=2, snip=5, channels=1024, height=8, width=8, seed=0, **block_kwargs):
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    block = TemporalContextBlock(inplanes=channels, **block_kwargs).to(device)
    block.eval()

    x = torch.randn(batch * snip, channels, height, width, device=device)
    print("=" * 80)
    print(f"block_kwargs={block_kwargs}")
    print(f"input x shape: {tuple(x.shape)}  # [B*S, C, H, W]")
    print(f"batch={batch}, snip={snip}, channels={channels}, height={height}, width={width}, device={device}")

    with torch.no_grad():
        y = block(x, snip)

    print(f"final output shape: {tuple(y.shape)}  # [B, C, H, W] when reduce=True, otherwise [B*S, C, H, W]-style aggregated output per frame")
    return y


## Example-use Configuration

This matches the `tcm_example_use.py` setup:

```python
TemporalContextBlock(inplanes=1024, window_size=5, repeat_mode=True, reduce=True)
```

In [6]:
_ = run_tcm_case(
    batch=2,
    snip=5,
    channels=1024,
    height=8,
    width=8,
    repeat_mode=True,
    window_size=5,
    reduce=True,
)


block_kwargs={'repeat_mode': True, 'window_size': 5, 'reduce': True}
input x shape: (10, 1024, 8, 8)  # [B*S, C, H, W]
batch=2, snip=5, channels=1024, height=8, width=8, device=cuda
 i am in mode2
input x size torch.Size([10, 1024, 8, 8]) claimed to be [B*S,C,H,W]
indentity size torch.Size([2, 5, 1024, 8, 8]) claimed to be [B,S,C,H,W]
indentity size after selecting center frame torch.Size([2, 1024, 8, 8]) claimed to be [B,C,H,W]
x size after view torch.Size([2, 5, 1024, 8, 8]) claimed to be [B,S,C,H,W]
feature_maps = x size torch.Size([2, 5, 1024, 8, 8]) claimed to be [B,S,C,H,W]
seprate_conv_stack conv 2, feature_maps size torch.Size([2, 4, 1024, 8, 8]) claimed to be [B,S,C,H,W], main size torch.Size([2, 1024, 8, 8]) claimed to be [B,C,H,W]
feature_maps size torch.Size([2, 4, 1024, 8, 8]) claimed to be [B,S,C,H,W]
main size torch.Size([2, 1024, 8, 8]) claimed to be [B,C,H,W]
step2so size torch.Size([2, 4, 1024, 8, 8]) claimed to be [B,S,C,H,W]
step2sc size torch.Size([2, 1, 1024, 8, 8

## Optional: Windowed Positional Mode

Run this cell too if you want to inspect the `mode1` path with positional encoding logic.

In [ ]:
_ = run_tcm_case(
    batch=1,
    snip=5,
    channels=64,
    height=8,
    width=8,
    repeat_mode=False,
    is_position_encoding=True,
    window_size=5,
    reduce=False,
)
